# Sudoku-3 : Resolution par Algorithme Genetique (C#)

**Navigation** : [<< Sudoku-2 DancingLinks C#](Sudoku-2-DancingLinks-Csharp.ipynb) | [Index](README.md) | [Sudoku-4 Simulated Annealing C# >>](Sudoku-4-SimulatedAnnealing-Csharp.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Comprendre les principes fondamentaux des algorithmes genetiques (selection, croisement, mutation)
2. Implementer une fonction de fitness pour evaluer la qualite d'une solution Sudoku
3. Concevoir differentes representations chromosomiques pour un probleme de contraintes
4. Analyser les performances et comparer les approches genetiques

**Voir aussi** : [Search-5-GeneticAlgorithms](../Search/Foundations/Search-5-GeneticAlgorithms.ipynb) pour la theorie des algorithmes genetiques

### Prerequis
- [Sudoku-2-DancingLinks-Csharp](Sudoku-2-DancingLinks-Csharp.ipynb) - comprehension des structures de donnees Sudoku
- Notions de base en biologie evolutive (selection naturelle, mutation)
- Programmation C# avancee (interfaces, generics, LINQ)

### Duree estimee : 45 minutes

---

## Introduction Theorique

Les algorithmes genetiques (GA) sont des techniques de recherche heuristiques inspirees par le processus de selection naturelle. Ils sont couramment utilises pour resoudre des problemes d'optimisation et de recherche. Un algorithme genetique utilise des operations telles que la mutation, le croisement et la selection pour evoluer vers une solution optimale.

Dans le contexte du Sudoku, un algorithme genetique peut etre utilise pour trouver une solution en representant une grille de Sudoku comme un chromosome, ou chaque gene represente une cellule de la grille. Les operations genetiques sont appliquees pour optimiser la grille en respectant les contraintes du Sudoku.

## Installation de GeneticSharp

Nous devons commencer par installer le package GeneticSharp.
Afin de concevoir un solver de Sudoku en algorithme genetique, il nous faudra concevoir un chromosome de Sudoku representant l'individu d'une population de solution potentielles, et une fonction d'evaluation (fitness) evaluant la qualite d'un individu/chromosome.
GeneticSharp fournira tout le reste des composants necessaires a la mise en oeuvre de l'algorithme.

In [ ]:
#r "nuget: GeneticSharp"

Installing Packages GeneticSharp

## Importation des Classes de Base

Nous allons importer les classes de base définies dans le notebook précédent, fournissant notamment la représentation, le chargement et l'affichage de Sudokus, et l'infrastructure de résolution.


In [ ]:
#!import Sudoku-0-Environment-Csharp.ipynb

# Notebook 0: Classes de Base pour la Résolution de Sudoku

Ce notebook contient les classes de base nécessaires pour la manipulation et la résolution des grilles de Sudoku. Il sera importé dans les autres notebooks pour réutiliser ces classes.

## Importation des Bibliothèques Nécessaires

Nous commençons par importer les bibliothèques nécessaires.


Installing Packages Plotly.NET

## Définition de la classe SudokuGrid

Nous définissons ici la classe SudokuGrid qui représente une grille de Sudoku et fournit des méthodes pour manipuler et afficher les grilles.


## Définition de l'interface ISudokuSolver

Nous définissons ici l'interface ISudokuSolver qui sera implémentée par les différentes stratégies de résolution de Sudoku.


## Définition de la classe SudokuHelper

Nous ajoutons ici la classe SudokuHelper qui contient des méthodes utilitaires pour charger  des grilles de Sudoku et tester des solvers.

- `GetSudokus` : Renvoie des listes de Sudoku issues de fichiers de 3 difficultés différentes.
- `SolveSudoku` : effectue un test simple d'un solver sur un sudoku donné.
- `TestSolvers` : exécute les tests de performance sur plusieurs solveurs.
- `DisplayResults` : affiche les résultats des tests sous forme de graphiques.



## Implémentation du Solveur de Sudoku par Algorithme Génétique

Nous allons maintenant implémenter ce solveur en C#. Comme indiqué précédemment il nous faut code la notion de chromosome et de fonction d'évaluation.

Afin de nous donner la possibilité de tester plusieurs types de chromosome, on introduit une interface dédiée:

### Interface `ISudokuChromosome`


In [ ]:
using GeneticSharp;

public interface ISudokuChromosome: IChromosome
{
	SudokuGrid TargetSudoku { get; set; }
    SudokuGrid GetSolution();
    
}

### Classe `SudokuFitness`

On peut maintenant proposer une fonction d'évaluation basée sur cette interface.
Cette classe évalue un chromosome en fonction du nombre d'erreurs dans la grille de Sudoku générée.


In [ ]:
using GeneticSharp;

public class SudokuFitness : IFitness
{
    private SudokuGrid _targetSudokuGrid;

    public SudokuFitness(SudokuGrid targetSudokuGrid)
    {
        _targetSudokuGrid = targetSudokuGrid;
    }

    public double Evaluate(IChromosome chromosome)
    {
        var sudokuChromosome = (ISudokuChromosome)chromosome;
        var sudoku = sudokuChromosome.GetSolution();
        var nbErrors = sudoku.NbErrors(_targetSudokuGrid);
        return -nbErrors;
    }
}


### Classe `SudokuSolver`

Cette classe contient les méthodes pour résoudre le Sudoku en utilisant l'algorithme génétique.

Nous concevons dores et déjà une méthode de résolution qui supportera plusieurs types de chromosomes de Sudoku sur les bases de notre interface commune, et présentant des fonctionnalités avancées suivantes:

- Possibilité de fournir l'opérateur de croisement et de mutation (opérateurs uniformes par défaut)

- Utilisation du parallélisme. L'application des opérateurs génétiques à une population ainsi que l'évaluation de la population résultantes sont par nature parallélisables. GeneticSharp propose l'utilisation du parallélisme de tâche natif .Net pour l'utilisation optimale des coeurs de processeurs disponibles.

- Multi-Objectifs et ajustement de la taille de la population. L'algorithme se termine à l'issue d'une génération si le meilleur chromosome ne contient plus d'erreur, ou si sa fitness n'a pas évolué depuis un certain nombre de génération, signalant un possible effondrement de la diversité génétique sur un maximum local. Dans ce cas, l'algorithme est redémarré en doublant la taille de la population.   


In [ ]:
using GeneticSharp;
using Microsoft.DotNet.Interactive;
using System;
using System.Diagnostics;

public class SudokuGeneticSolver: ISudokuSolver
{
    
    public ISudokuChromosome Chromosome { get; set; }
    public IMutation Mutation { get; set; } = new UniformMutation(true);
    public ICrossover Crossover { get; set; } = new UniformCrossover(0.5f);


    public SudokuGeneticSolver(ISudokuChromosome chromosome)
    {
        Chromosome = chromosome;
    }

    

    public SudokuGrid Solve(SudokuGrid s)
    {
        var maxDuration = 30;
        var maxPopulation = 100000;
        var fitness = new SudokuFitness(s);
        
        var populationSize = 400;
        //Opérateur de sélection (Elite rapide mais un peu sélectif)
        var selection = new EliteSelection ();

        // Critères de terminaison
        var fitnessThreshold = 0;
        int stableGenerationNb = 30;
        var termination = new OrTermination(new ITermination[]
        {
            new FitnessThresholdTermination(fitnessThreshold),
            new FitnessStagnationTermination(stableGenerationNb),
        });
        
        var stopWatch = Stopwatch.StartNew();
        var lastTime = stopWatch.Elapsed;
        var displayPlaceholder = display("Initialisation...");
        var sudokuPlaceHloder = display(s.ToString());

        SudokuGrid bestSudoku = null;
        do
        {
            Population population = new Population(populationSize, populationSize, Chromosome);

            GeneticAlgorithm ga = new GeneticAlgorithm(population, fitness, selection, Crossover, Mutation)
            {
                Termination = termination,
                // Opérateurs de parallélisation
                OperatorsStrategy = new TplOperatorsStrategy(),
                TaskExecutor = new TplTaskExecutor(),
                //MutationProbability = 0.1f,
                // CrossoverProbability = 0.75f
            };
            
            ga.GenerationRan += (sender, args) =>
            {
                var bestIndividual = (ISudokuChromosome)ga.Population.BestChromosome;
                bestSudoku = bestIndividual.GetSolution();
                var nbErrors = bestSudoku.NbErrors(s);
                var message = $"Generation {ga.GenerationsNumber}, Population {ga.Population.CurrentGeneration.Chromosomes.Count}, NbErrors {bestSudoku.NbErrors(s)}, Elapsed {stopWatch.Elapsed - lastTime}";
                displayPlaceholder.Update(message);
                sudokuPlaceHloder.Update(bestSudoku.ToString());
            
            };
            lastTime = stopWatch.Elapsed;
            ga.Start();
            populationSize = Math.Min(maxPopulation, populationSize *= 2);
        } while (bestSudoku.NbErrors(s) > 0 && stopWatch.Elapsed.TotalSeconds < maxDuration);

        return bestSudoku;

    }
    
}

### Premier chromosome simple

Pour ce premier essai, nous représentont chaque cellule par un gène à valeur de 1 à 9. L'initialisation tient compte du masque à résoudre, mais pas les opérateurs qui agissent au hasard.

In [ ]:
using System;

public class SudokuCellsChromosome : ChromosomeBase, ISudokuChromosome
{
    public SudokuGrid TargetSudoku { get; set; }

    
    public SudokuCellsChromosome() : base(81) {}

    public override Gene GenerateGene(int geneIndex)
    {
        var row = geneIndex / 9;
        var col = geneIndex % 9;

        if (TargetSudoku.Cells[row, col] != 0)
        {
            return new Gene(TargetSudoku.Cells[row, col]);
        }

        var rnd = RandomizationProvider.Current;
        return new Gene(rnd.GetInt(1, 10));
    }

    public override IChromosome CreateNew()
    {
        var toReturn = new SudokuCellsChromosome(){TargetSudoku = TargetSudoku};
        toReturn.CreateGenes();
        return toReturn;
    }

    public SudokuGrid GetSolution()
    {
        var genes = GetGenes();
        var cells = new int[9, 9];
        for (int i = 0; i < 81; i++)
        {
            cells[i / 9, i % 9] = (int)genes[i].Value;
        }
        var sudoku = new SudokuGrid() { Cells = cells };
        return sudoku;
    }
}


## Test du Solveur

Nous allons maintenant tester notre solveur de Sudoku par algorithme génétique en utilisant des grilles de Sudoku de différentes difficultés : facile, moyen et difficile.


In [ ]:
display("Test du solver genetique simple:");


display("Puzzle Sudoku Facile Initial:");

// Charger et tester un puzzle facile
var easySudokus = SudokuHelper.GetSudokus(SudokuDifficulty.Easy);
var easySudoku = easySudokus.FirstOrDefault();

//Création du chromosome:
var chromosome = new SudokuCellsChromosome(){TargetSudoku = easySudoku};

// Instanciation de Solver avec notre chromosmome SudokuCellsChromosome
var solver = new SudokuGeneticSolver(chromosome);

SudokuHelper.SolveSudoku(easySudoku, solver);

Test du solver genetique simple:

Puzzle Sudoku Facile Initial:

Résolution par le solver SudokuGeneticSolver du Sudoku:
 -------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Initialisation...

-------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Sudoku renvoyé:
-------------------------------
| 9  6  2 | 1  8  5 | 4  7  3 | 
| 1  7  4 | 9  6  3 | 8  2  5 | 
| 5  3  8 | 4  2  7 | 1  6  9 | 
-------------------------------
| 8  2  6 | 3  5  9 | 7  4  1 | 
| 3  5  7 | 8  1  4 | 2  9  6 | 
| 4  9  1 | 6  7  2 | 5  3  8 | 
-------------------------------
| 2  4  9 | 5  3  8 | 6  1  7 | 
| 7  1  5 | 2  9  6 | 3  8  4 | 
| 6  8  3 | 7  4  1 | 9  5  2 | 
-------------------------------
Nombre d'erreurs réstantes: 0
Temps de résolution: 5733,8035 ms

### Interpretation des resultats

Le solveur genetique simple avec chromosome `SudokuCellsChromosome` resout le Sudoku facile :

| Aspect | Observation | Signification |
|--------|-------------|---------------|
| Resolution | Sudoku facile resolu | L'algorithme fonctionne mais necessite plusieurs redemarrages |
| Performance | Plusieurs secondes | L'approche n'est pas optimale pour les difficultes elevees |
| Population | Ajustement dynamique | La taille de population double en cas de stagnation |

**Points cles** :
1. La representation chromosomique influence fortement la performance
2. Les operateurs uniformes ne sont pas adaptes aux problemes de contraintes
3. Le redemarrage avec population augmentee permet de sortir des maxima locaux

> **Note technique** : Ce premier resultat montre le potentiel des algorithmes genetiques mais aussi leurs limites. Nous allons maintenant explorer une representation plus intelligente basee sur les permutations.


## Conclusion Intermédiaire

Le solveur génétique a réussi à résoudre le Sudoku de difficulté facile en plusieurs redémarrages, prenant plusieurs secondes. Cela montre que notre approche fonctionne, mais elle n'est pas encore assez efficace pour résoudre des puzzles de difficulté plus difficile en un temps raisonnable.


### Chromosome : SudokuPermutationsChromosome

Le SudokuPermutationsChromosome est un chromosome simple de 9 gènes qui manipule les permutations de lignes du Sudoku. Chaque gène représente une permutation d'une ligne entière, permettant ainsi une exploration plus efficace des solutions potentielles.


In [ ]:
public class SudokuPermutationsChromosome  : ChromosomeBase, ISudokuChromosome
{

     public SudokuGrid TargetSudoku { get; set; }
    protected static IList<IList<int>> allPermutations = GetPermutations(Enumerable.Range(1, 9), 9).ToList();

    private readonly IList<IList<int>>[] _rowPermutationsCache;

    public SudokuPermutationsChromosome(SudokuGrid targetSudoku) 
        : base(9)
    {
        TargetSudoku = targetSudoku;
        _rowPermutationsCache = new IList<IList<int>>[9];
        CreateGenes();
    }

    private SudokuPermutationsChromosome(SudokuGrid targetSudoku, IList<IList<int>>[] rowPermutationsCache)
        : base( 9)
    {
         TargetSudoku = targetSudoku;
        _rowPermutationsCache = rowPermutationsCache;
        CreateGenes();
    }

    public override Gene GenerateGene(int geneIndex)
    {
        var rowPermutations = GetRowPermutations(geneIndex);
        var rnd = RandomizationProvider.Current;
        var chosenPermutation = rowPermutations[rnd.GetInt(0, rowPermutations.Count)];
        return new Gene(chosenPermutation);
    }

    private IList<IList<int>> GetRowPermutations(int row)
    {
        if (_rowPermutationsCache[row] == null)
        {
            _rowPermutationsCache[row] = allPermutations
                .Where(perm => IsValidPermutation(perm, row)).ToList();
        }
        return _rowPermutationsCache[row];
    }

    private static IList<IList<int>> GetPermutations(IEnumerable<int> list, int length)
    {
        if (length == 1) return list.Select(t => (IList<int>)(new List<int> { t })).ToList();
        return GetPermutations(list, length - 1)
            .SelectMany(t => list.Where(e => !t.Contains(e)),
                (t1, t2) => (IList<int>)t1.Concat(new List<int> { t2 }).ToList()).ToList();
    }

    private bool IsValidPermutation(IList<int> permutation, int row)
    {
        for (int col = 0; col < 9; col++)
        {
            if (TargetSudoku.Cells[row, col] != 0 && TargetSudoku.Cells[row, col] != permutation[col])
            {
                return false;
            }
        }
        return true;
    }

    public override IChromosome CreateNew()
    {
        return new SudokuPermutationsChromosome(TargetSudoku, _rowPermutationsCache);
    }

    public SudokuGrid GetSolution()
    {
        var newGrid = (SudokuGrid)TargetSudoku.Clone();
        var genes = GetGenes();
        for (int row = 0; row < 9; row++)
        {
            var permutation = (IList<int>)genes[row].Value;
            for (int col = 0; col < 9; col++)
            {
                newGrid.Cells[row, col] = permutation[col];
            }
        }
        return newGrid;
    }
}


### Test du SudokuPermutationsChromosome

Nous allons maintenant tester le SudokuPermutationsChromosome en utilisant notre algorithme génétique. Nous allons voir comment il se comporte sur un Sudoku de difficulté facile et moyenne.

In [ ]:
display("Test du solver genetique de permutations de lignes:");


display("Puzzle Sudoku Facile Initial:");

// Charger et tester un puzzle facile
var easySudokus = SudokuHelper.GetSudokus(SudokuDifficulty.Easy);
var easySudoku = easySudokus.FirstOrDefault();

//Création du chromosome:
var chromosome = new SudokuPermutationsChromosome(easySudoku);

// Instanciation de Solver avec notre chromosmome SudokuCellsChromosome
var solver = new SudokuGeneticSolver(chromosome);

SudokuHelper.SolveSudoku(easySudoku, solver);

display("Puzzle Sudoku Medium Initial:");

// Charger et tester un puzzle moyen
var mediumSudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Medium).First();

chromosome = new SudokuPermutationsChromosome(mediumSudoku);
solver = new SudokuGeneticSolver(chromosome);

SudokuHelper.SolveSudoku(mediumSudoku, solver);

Test du solver genetique de permutations de lignes:

Puzzle Sudoku Facile Initial:

Résolution par le solver SudokuGeneticSolver du Sudoku:
 -------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

Initialisation...

-------------------------------
| 9     2 |       5 | 4     3 | 
| 1       |    6  3 |    2  5 | 
| 5     8 | 4     7 |    6    | 
-------------------------------
|    2  6 | 3     9 |       1 | 
|    5  7 |    1    | 2  9    | 
|    9    | 6  7    | 5  3    | 
-------------------------------
| 2  4    | 5  3    | 6       | 
| 7     5 | 2       | 3     4 | 
|    8    |    4  1 | 9  5    | 
-------------------------------

...

Puzzle Sudoku Medium Initial:

Résolution par le solver SudokuGeneticSolver du Sudoku:
 -------------------------------
| 8  5    |       2 | 4       | 
| 7  2    |         |       9 | 
|       4 |         |         | 
-------------------------------
|         | 1     7 |       2 | 
| 3     5 |         | 9       | 
|    4    |         |         | 
-------------------------------
|         |    8    |    7    | 
|    1  7 |         |         | 
|         |    3  6 |    4    | 
-------------------------------

Initialisation...

-------------------------------
| 8  5    |       2 | 4       | 
| 7  2    |         |       9 | 
|       4 |         |         | 
-------------------------------
|         | 1     7 |       2 | 
| 3     5 |         | 9       | 
|    4    |         |         | 
-------------------------------
|         |    8    |    7    | 
|    1  7 |         |         | 
|         |    3  6 |    4    | 
-------------------------------

Sudoku renvoyé:
-------------------------------
| 8  5  1 | 3  9  2 | 4  6  7 | 
| 7  2  3 | 4  6  5 | 1  8  9 | 
| 6  9  4 | 8  1  7 | 5  2  3 | 
-------------------------------
| 9  6  8 | 1  4  7 | 3  5  2 | 
| 3  7  5 | 6  2  8 | 9  1  4 | 
| 1  4  2 | 9  5  3 | 7  6  8 | 
-------------------------------
| 4  3  6 | 2  8  1 | 9  7  5 | 
| 2  1  7 | 5  4  9 | 8  3  6 | 
| 5  8  9 | 7  3  6 | 2  4  1 | 
-------------------------------
Nombre d'erreurs réstantes: 4
Temps de résolution: 44979,1705 ms

### Interpretation des resultats

Le solveur genetique base sur les permutations de lignes montre des performances differentes selon la difficulte :

| Difficulte | Resultat attendu | Observations |
|------------|------------------|--------------|
| Facile | Resolution rapide | Convergence en quelques generations |
| Moyen | Resolution possible | Plus de generations necessaires |

**Points cles** :
1. L'approche par permutations respecte les contraintes de lignes, reduisant l'espace de recherche
2. La fitness negative (nombre d'erreurs) permet de guider l'evolution vers la solution
3. Le parallelimse accelere l'evaluation de la population

> **Note technique** : Les algorithmes genetiques sont bien adaptes aux problemes d'optimisation ou l'espace de solutions est trop grand pour une recherche exhaustive.


## Conclusion Finale

Bien que notre approche avec le SudokuPermutationsChromosome montre une amélioration par rapport au chromosome simple, il reste des défis importants pour résoudre des puzzles de difficulté moyenne et élevée. 

Le principal problème rencontré semble être un effondrement de la diversité génétique, même avec l'utilisation des permutations. Le problème du Sudoku est difficile d'accès aux algorithmes génétiques du fait notamment que l'espace de solution présente de nombreux extrema locaux sur lesquels ils restent coincés, et une très faible densité de solution.

Dans les notebooks suivants, nous explorerons d'autres méthodes et approches pour résoudre les puzzles de Sudoku, en espérant trouver des solutions plus efficaces et rapides.

### Pour aller plus loin

Pour améliorer l'efficacité des algorithmes génétiques dans la résolution de Sudoku, il serait essentiel d'explorer des opérateurs plus performants que les croisements et mutations uniformes actuellement utilisés. 

Des opérateurs dits "ordered" issus de la littérature génétique, pourraient offrir des gains significatifs. Ces opérateurs, basés sur des découpages et recollages inspirés de la génétique, permettent des transformations de type permutations sur des gènes présentés comme une liste réordonnable. Utilisés dans le problème du voyageur de commerce (TSP), ils améliorent sensiblement les qualités de la convergence.

 Appliqués au Sudoku, ils pourraient permettre la recombinaison de "bouts de bonnes solutions" pour converger plus efficacement vers la solution optimale. Mais cela impliquerait tout d'abord de géométriser le chromosome du Sudoku pour permettre leur bon fonctionnement.

---

**Navigation** : [<< Sudoku-2 DancingLinks C#](Sudoku-2-DancingLinks-Csharp.ipynb) | [Index](README.md) | [Sudoku-4 Simulated Annealing C# >>](Sudoku-4-SimulatedAnnealing-Csharp.ipynb)

**Voir aussi**: 
- [Search-5-GeneticAlgorithms](../Search/Foundations/Search-5-GeneticAlgorithms.ipynb) - Theorie des algorithmes genetiques
- [Search-App-9-EdgeDetection](../Search/Applications/Search-App-9-EdgeDetection.ipynb) - Autre application des GA

## Exercice : Chromosome par Permutations de Blocs 3x3

### Enonce

Jusqu'ici, le chromosome `SudokuPermutationsChromosome` travaille avec des permutations de lignes, garantissant les contraintes de lignes. Explorez une alternative basee sur les blocs 3x3.

Implementez `SudokuBlockChromosome` ou chaque gene represente une permutation des valeurs manquantes dans un bloc 3x3 :
1. Identifiez les valeurs fixes et manquantes dans chaque bloc
2. Generez uniquement les permutations compatibles avec les valeurs deja presentes
3. Testez votre chromosome sur un puzzle facile et comparez avec `SudokuPermutationsChromosome`

### Indice

Un bloc 3x3 peut avoir de 0 a 9 valeurs fixes. Si un bloc a k valeurs fixes, il existe (9-k)! permutations des valeurs manquantes. Cette representation garantit les contraintes de blocs (au lieu des lignes), laissant les contraintes de lignes et colonnes a la fonction de fitness.

In [ ]:
// EXERCICE : SudokuBlockChromosome - Permutations de blocs 3x3
// TODO: Implementez un chromosome base sur des permutations de blocs

using GeneticSharp;

public class SudokuBlockChromosome : ChromosomeBase, ISudokuChromosome
{
    public SudokuGrid TargetSudoku { get; set; }
    
    // Cache des permutations valides pour chaque bloc (index 0-8)
    private IList<IList<int>>[] _blockPermutationsCache;
    
    public SudokuBlockChromosome(SudokuGrid targetSudoku) : base(9)
    {
        TargetSudoku = targetSudoku;
        _blockPermutationsCache = new IList<IList<int>>[9];
        CreateGenes();
    }
    
    private IList<IList<int>> GetBlockPermutations(int blockIndex)
    {
        if (_blockPermutationsCache[blockIndex] == null)
        {
            // TODO: Calculer les permutations valides pour le bloc blockIndex
            // Etape 1: Determiner la position du bloc (blockRow = blockIndex / 3, blockCol = blockIndex % 3)
            // Etape 2: Identifier les valeurs fixes et manquantes dans le bloc
            // Etape 3: Generer toutes les permutations des valeurs manquantes
            // Etape 4: Reconstruire la liste complete de 9 valeurs pour chaque permutation
            throw new NotImplementedException("A vous de jouer !");
        }
        return _blockPermutationsCache[blockIndex];
    }
    
    public override Gene GenerateGene(int geneIndex)
    {
        var perms = GetBlockPermutations(geneIndex);
        var rnd = RandomizationProvider.Current;
        return new Gene(perms[rnd.GetInt(0, perms.Count)]);
    }
    
    public override IChromosome CreateNew()
    {
        return new SudokuBlockChromosome(TargetSudoku);
    }
    
    public SudokuGrid GetSolution()
    {
        var grid = (SudokuGrid)TargetSudoku.Clone();
        var genes = GetGenes();
        
        // TODO: Reconstruire la grille depuis les genes
        // Chaque gene[blockIndex] est une permutation de 9 valeurs pour le bloc correspondant
        // Replacer les valeurs dans les bonnes cellules
        throw new NotImplementedException("A vous de jouer !");
    }
}

// Test de votre implementation (decommentez quand SudokuBlockChromosome est implemente)
// var easySudoku = SudokuHelper.GetSudokus(SudokuDifficulty.Easy).First();
// var blockChromosome = new SudokuBlockChromosome(easySudoku);
// var solver = new SudokuGeneticSolver(blockChromosome);
// SudokuHelper.SolveSudoku(easySudoku, solver);

Console.WriteLine("TODO: Implementez SudokuBlockChromosome pour tester");